In [1]:
import numpy as np
import sklearn as sk
import pandas as pd

In [2]:
df = pd.read_csv('cattle_data_train.csv')
print(len(df))


210000


Data Cleaning

In [3]:
labels = df["Milk_Yield_L"]

data = df.drop(columns=["Cattle_ID","Date","Farm_ID","Milk_Yield_L"])

# data = data.dropna() # Just drop all NaNs for now
# print(data.head())

categorical_cols = [c for c in data.columns if data[c].dtype == 'object']
numeric_cols = [c for c in data.columns if c not in categorical_cols]

# print(categorical_cols)
# print(numeric_cols)

selected_cols = categorical_cols[:]+numeric_cols[:]
selected_cols.append("Milk_Yield_L")
potential_data = df[selected_cols]
temp = sk.compose.ColumnTransformer(transformers=[("categorical",sk.preprocessing.OneHotEncoder(handle_unknown='ignore'),categorical_cols)],remainder="passthrough")
cpy = temp.fit_transform(potential_data)
print(pd.DataFrame(cpy,columns=temp.get_feature_names_out()).corr()["remainder__Milk_Yield_L"])




categorical__Breed_ Brown Swiss                 -0.001673
categorical__Breed_Brown Swiss                  -0.001069
categorical__Breed_Brown Swiss                  -0.002514
categorical__Breed_Guernsey                     -0.000333
categorical__Breed_Holstein                      0.001445
categorical__Breed_Holstien                      0.001963
categorical__Breed_Jersey                       -0.000510
categorical__Climate_Zone_Arid                   0.002194
categorical__Climate_Zone_Continental           -0.000920
categorical__Climate_Zone_Mediterranean         -0.001282
categorical__Climate_Zone_Subtropical           -0.000215
categorical__Climate_Zone_Temperate             -0.000323
categorical__Climate_Zone_Tropical               0.000545
categorical__Management_System_Extensive         0.002045
categorical__Management_System_Intensive         0.000791
categorical__Management_System_Mixed            -0.001226
categorical__Management_System_Pastoral         -0.000370
categorical__M

Remove features which have little correlation with milk yield

In [ ]:
# Dropping all columns with less than .01 correlation with Milk_Yield_L
remove_cols = ["Breed","Climate_Zone","Management_System","Feed_Type","Feeding_Frequency","Walking_Distance_km",
               "Grazing_Duration_hrs","Resting_Hours","Humidity_percent","Housing_Score","FMD_Vaccine","Brucellosis_Vaccine"
               ,"HS_Vaccine","BQ_Vaccine","BVD_Vaccine","Body_Condition_Score"]

categorical_cols=[x for x in categorical_cols if x not in remove_cols]
numeric_cols=[x for x in numeric_cols if x not in remove_cols]
print(categorical_cols)
print(numeric_cols)

['Lactation_Stage']
['Age_Months', 'Weight_kg', 'Parity', 'Days_in_Milk', 'Feed_Quantity_kg', 'Water_Intake_L', 'Rumination_Time_hrs', 'Ambient_Temperature_C', 'Anthrax_Vaccine', 'IBR_Vaccine', 'Rabies_Vaccine', 'Previous_Week_Avg_Yield', 'Milking_Interval_hrs', 'Feed_Quantity_lb', 'Mastitis']


In [4]:
numeric_transformer = sk.pipeline.Pipeline(steps=[
    ('imputer', sk.impute.SimpleImputer(strategy='median')),
    ('scaler', sk.preprocessing.StandardScaler())
])


categorical_transformer = sk.pipeline.Pipeline(steps=[
    ('imputer', sk.impute.SimpleImputer(strategy='most_frequent')),
    ('onehot', sk.preprocessing.OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = sk.compose.ColumnTransformer(transformers=[("numeric",numeric_transformer,numeric_cols),
("categorical",categorical_transformer,categorical_cols)])
# TODO: try ridge as well, as well as trees
pca = sk.decomposition.PCA()
bag = sk.ensemble.BaggingRegressor(estimator=sk.svm.SVR())
pipeline = sk.pipeline.Pipeline(steps=[("pre",preprocessor),("pca",pca),("bag",bag)])

score = sk.model_selection.cross_val_score(pipeline,data,labels,scoring="neg_root_mean_squared_error",verbose=3,n_jobs=-5)
print(score)

[Parallel(n_jobs=-5)]: Using backend LokyBackend with 16 concurrent workers.


KeyboardInterrupt: 

Build model with best params

In [ ]:
test = pd.read_csv('cattle_data_test.csv')
inputs = test.drop(columns=["Cattle_ID","Date","Farm_ID"])
predictions = model.predict(inputs)
test["Milk_Yield_L"]=predictions
test[["Cattle_ID","Milk_Yield_L"]].to_csv("./results.csv",index=False)

In [ ]:
import pickle
pickle.dump(model,open("./final_model.sav","wb"))